# The K-Armed Bandit Problem

Thi first assignament of this section is to read Chap. 2-2.7 of *Reinforcement Learning: An Introduction* by Sutton.

## Chapter 2: Multi-Armed Bandits

The feature that distinguish RL from other learning types is that its trained by *evaluating* actions taken rather than from *intructs* by givin correct actions, creating the need for active exploration.

### 2.1 - A *k*-armed Bandit Problem

**Problem:** You are faced repeatedly with a choice among *k* different options or actions. After each choice you recieve a numerical reward chosen from a stationary probability distribution that depends on the action you selected. Your objective is to maximize the expected total reward over some time period.

Each of the *k* actions has an expected or mean reward given that an action $a$ is selected, called the *value* of that action, functionally seen as,

$$q_* (a) = \mathbb{E}[R_t | A_t = a]$$

where $A_t$ is the action selected on time step t, and the corresponding reward as $R_t$. If one knows the value of each action, the problem is trivial (choose action with highest value). If we don't know, but have estimates, we denote the estimated value of action $a$ at time stem $t$ as $Q_t (a)$, and want it to be close to $q_* (a)$.

If you mantain estimates of the action values, then at any time step there is at least one action whose estimated value is greatest. We call these the *greedy* actions. When you select one of these actions, we say that you are *exploiting* your current knowledge of the values of the actions. If instead you select one of the nongreedy actions, then we say you are *exploring*. Exploitation is the right thing to do to maximize the expected reward on the one step, but exploration may produce the greater total reward in the long run. In either way, it is important to find a way to balance the exploitation and the exploration in order to find the best $Q$.

### 2.2 - Action-value Methods

In methods where you estimates the values of actions and use those estimations to make action selection decisions are called *action-value methods*. One natural way to estimate the value is by averaging the rewards actually received,

$$Q_t (a) = \frac{\text{sum of rewards when $a$ taken prior to $t$}}{\text{number of times $a$ taken prior to $t$}} = \frac{\sum_{i=1}^{t-1} R_i \cdot \mathbb{1}_{A_i = a}}{\sum_{i=1}^{t-1}\mathbb{1}_{A_i = a}},$$

where $\mathbb{1}_{predicate}$ denotes the random variable that is 1 if $predicate$ is true and $0$ if it is not. If the denominator is 0, then define $Q_t (a)$ as some default value, such as 0. As the denominator goes to infinity, by the law of large numbers, $Q_t (a)$ converges to $q_* (a)$. This is the *sample-average* method for estimating action values.

The simplest action selection rule is the *greedy* action selection method,

$$A_t = \text{arg max}_a Q_t (a),$$

where $\text{arg max}_a$ denotes the action $a$ for which the expression that follows is maximized. This method exploits the current knowledge to maximize immediate reward. A simple way to combine this with exploration is that, with probability $\epsilon$, instead of choosing the greedy action, select randomly from among all the actions with equal probability, independently of $Q_t$. This is called the $\epsilon$-*greedy* methods.

### 2.3 - The 10-armed Testbed

TO-DO: Do the example here

### 2.4 - Incremental Implementation

Let $R_i$ denote the reward received after $i$th selection of *this action* and let $Q_n$ the estimate of its action value after it has been selected $n-1$ times, then,

$$Q_n = \frac{R_1 + R_2 + \cdots + R_{n-1}}{n-1}.$$

The obvious implementation is mantain a record of all the rewards and compute $Q_n$ whenever is needed, but this have memory and computational issues on the long run. Each additional reward requires additional memory to store and additional computation to compute the sum.

To mitigate this problems, given $Q_n$ and the $n$-th reward $R_n$,

$\hspace{9cm}Q_{n+1} = \frac{1}{n} \sum_{i=1}^n R_i \\ 
\hspace{9.87cm} = \frac{1}{n} \left(R_n + \sum_{i=1}^{n-1} R_i\right) \\
\hspace{9.87cm} = \frac{1}{n} \left(R_n + (n-1)\frac{1}{n-1}\sum_{i=1}^{n-1} R_i\right)\\
\hspace{9.87cm} = \frac{1}{n} \left(R_n + (n-1) Q_n\right)\\ 
\hspace{9.87cm} = \frac{1}{n} \left(R_n + nQ_n - Q_n\right) \\
\hspace{9.87cm} = Q_n + \frac{1}{n}\left[R_n - Q_n\right]$

This implementation requires memory only for $Q_n$ and $n$. This update is of a form:

$$NewEstimate \hspace{0.3cm} \leftarrow \hspace{0.3cm} OldEstimate + StepSize \hspace{0.1cm}[Target - OldEstimate]$$

Here is pseudo-code for a simple bandit algorithm:

1. Initialize, for $a=1$ to $k$: $Q(a) \rightarrow 0$ and $N(a) \rightarrow 0$.
2. Loop forever:
    - $A \rightarrow \begin{cases} argmax_a Q(a) \hspace{0.93cm} \text{with probability } 1-\epsilon \text{ (breaking ties randomly)} \\ \text{a random action} \hspace{0.5cm} \text{ with probability } \epsilon \end{cases}$
    - $R \rightarrow bandit(A)$
    - $N(A) \rightarrow N(A) + 1$
    - $Q(A) \rightarrow Q(A) + \frac{1}{N(A)}[R - Q(A)]$


### 2.5 - Tracking a Nonstationary Problem

In general, RL problems are nonstationary, that is, the reward probabilities change over time. In such cases, it makes sense to give more weight to recent rewards than to long-past rewards. One way to do this is to use a constant step-size parameter:

$\hspace{9cm}Q_{n+1} = Q_n + \alpha[R_n - Q_n] \\
\hspace{9.87cm} = \alpha R_n + (1 - \alpha) Q_n \\
\hspace{9.87cm} = \alpha R_n + (1 - \alpha) [\alpha R_{n-1} + (1-\alpha) Q_{n-1}]\\
\hspace{9.87cm} = \alpha R_n + (1 - \alpha) \alpha R_{n-1} + (1-\alpha)^2 \alpha R_{n-2} + \\
\hspace{13.87cm} \cdots + (1 - \alpha)^{n-1}\alpha R_1 + (1-\alpha)^n Q_1 \\
\hspace{9.87cm} = (1 - \alpha)^n Q_1 + \sum_{i=1}^n \alpha (1-\alpha)^{n-i} R_i.$

We call this a weighted average, or exponential recency-weighted average.

Let $\alpha_n (a)$ denote the step-size parameter used to process the reward received after de $n$th selection of action $a$. A well-known result in stochastic approx. theory gives us the tonditions requider to assure convergence with prob. 1:

$$\sum_{n=1}^\infty \alpha_n(a) = \infty \hspace{0.2cm} \text{ and } \hspace{0.2cm} \sum_{n=1}^\infty \alpha_n^2 (a) < \infty.$$

### 2.6 - Optimistic Initial Values

All the methods discused above are *biased*, since it depends on $Q_1 (a)$. Any method that focuses on the initial conditions in any special way is unlikely to help with the general nonstationary case, since it only enhace the exploration in the beggining of the training,and if the task changes in time, creating renewed need of exploration, this methods cannot help.

### 2.7 - Upper-Confidence-Bound Action Selection

$\epsilon$-greedy action selecton explores non-greedy actions with no preference. It would be better to select among the non-greedy actions according to their potential for actually being optimal. One effective way to do this is,

$$A_t = \text{arg max}_a \left[Q_t (a) + c\sqrt{\frac{\ln t}{N_t (a)}}\right],$$

where the number $c>0$ controls the degree of exploration. If $N_t (a) = 0$, then $a$ is considered to be a maximizing action.

The idea of this *upper confidence bound* (UCB) action selection is that the square-root term is a measure of the uncertainty or variance in the estimate of $a$'s value.